# Streaming Responses for Real-Time Interaction

## Overview

This notebook demonstrates how to implement streaming responses in agentic AI systems for real-time, interactive user experiences.

## Learning Objectives

- Understand streaming vs batch response patterns
- Implement token-by-token streaming with LangChain
- Build real-time conversation interfaces
- Handle streaming errors gracefully
- Optimize streaming performance

## Exam Objectives: 2.5
## Estimated Time: 60-75 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Error handling patterns (retry logic, circuit breakers)
- ⭐⭐⭐ Prompt engineering techniques
- ⭐⭐ Tool integration best practices
- ⭐⭐ Streaming responses

**Common Exam Scenarios:**
- Implementing retry logic for API failures
- Designing prompts for complex tasks
- Handling tool execution errors

**Key Concepts to Master:**
- Exponential backoff for retries
- Circuit breaker pattern
- Dynamic prompt chains
- Graceful degradation

## Setup: Import Dependencies

In [ ]:
# Import core libraries for streaming
import os
import sys
import time
import asyncio
from typing import Iterator, AsyncIterator, Dict, Any, Optional
from datetime import datetime

# LangChain imports for streaming
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.callbacks.base import BaseCallbackHandler

# Verify LangChain is available
try:
    import langchain
    print(f"✅ LangChain version: {langchain.__version__}")
except ImportError as e:
    print(f"❌ LangChain not available: {e}")

print("🎯 Setup complete!")

## Theory: Streaming vs Batch Responses

### Why Streaming?

**Streaming** delivers responses incrementally as they're generated, providing:
- **Better UX**: Users see progress immediately
- **Lower perceived latency**: First token arrives quickly
- **Interruptibility**: Users can stop generation early
- **Real-time feel**: More conversational interaction

### Streaming vs Batch Comparison

| Aspect | Batch | Streaming |
|--------|-------|----------|
| **Time to First Token** | High (wait for full response) | Low (immediate) |
| **Total Time** | Same | Same |
| **User Experience** | Waiting... | Progressive display |
| **Complexity** | Simple | More complex |
| **Use Cases** | Batch processing, APIs | Chat interfaces, real-time apps |

### Streaming Architecture

```
LLM → Token Generator → Stream Buffer → Callback Handler → UI Display
      (generates)       (buffers)       (processes)       (renders)
```

### Key Concepts

1. **Token**: Smallest unit of text (word, subword, or character)
2. **Callback**: Function called for each token
3. **Buffer**: Temporary storage for tokens
4. **Flush**: Send buffered tokens to output

## Implementation: Basic Streaming

Let's implement basic token-by-token streaming.

In [ ]:
# Simulated streaming LLM for demonstration
class MockStreamingLLM:
    """
    Mock LLM that simulates streaming token generation.
    
    In production, use real streaming LLMs like:
    - ChatNVIDIA with streaming=True
    - OpenAI with stream=True
    - Anthropic with stream=True
    """
    
    def __init__(self, delay_per_token: float = 0.05):
        """
        Initialize mock streaming LLM.
        
        Args:
            delay_per_token: Simulated delay between tokens (seconds)
        """
        self.delay_per_token = delay_per_token
    
    def stream(self, prompt: str) -> Iterator[str]:
        """
        Stream tokens one at a time.
        
        Args:
            prompt: Input prompt
        
        Yields:
            Individual tokens
        """
        # Simulated response
        response = "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation. It works by first retrieving relevant documents from a knowledge base, then using those documents as context for generating responses."
        
        # Split into tokens (words for simplicity)
        tokens = response.split()
        
        # Stream tokens one at a time
        for i, token in enumerate(tokens):
            # Simulate generation delay
            time.sleep(self.delay_per_token)
            
            # Yield token with space (except last token)
            if i < len(tokens) - 1:
                yield token + " "
            else:
                yield token

# Test basic streaming
print("=== Basic Streaming Demo ===")
print("Prompt: What is RAG?\n")
print("Response (streaming):")

llm = MockStreamingLLM(delay_per_token=0.05)
start_time = time.time()

# Stream and display tokens
for token in llm.stream("What is RAG?"):
    print(token, end='', flush=True)  # flush=True ensures immediate display

elapsed = time.time() - start_time
print(f"\n\nTotal time: {elapsed:.2f}s")
print(f"Time to first token: ~{llm.delay_per_token:.2f}s")

## Implementation: Custom Streaming Callback

Build custom callbacks to handle streaming tokens.

In [ ]:
# Custom streaming callback handler
class CustomStreamingCallback(BaseCallbackHandler):
    """
    Custom callback handler for streaming responses.
    
    Features:
    - Token counting
    - Latency tracking
    - Custom formatting
    - Error handling
    """
    
    def __init__(self, prefix: str = "", show_metrics: bool = True):
        """
        Initialize callback handler.
        
        Args:
            prefix: Prefix to add before each token
            show_metrics: Whether to show metrics after streaming
        """
        self.prefix = prefix
        self.show_metrics = show_metrics
        
        # Metrics tracking
        self.token_count = 0
        self.start_time = None
        self.first_token_time = None
        self.tokens = []
    
    def on_llm_start(self, *args, **kwargs):
        """
        Called when LLM starts generating.
        
        Initialize timing metrics.
        """
        self.start_time = time.time()
        self.token_count = 0
        self.tokens = []
        print(f"{self.prefix}[Streaming started]\n", flush=True)
    
    def on_llm_new_token(self, token: str, **kwargs):
        """
        Called for each new token.
        
        Args:
            token: New token generated
        """
        # Track first token latency
        if self.first_token_time is None:
            self.first_token_time = time.time()
        
        # Update metrics
        self.token_count += 1
        self.tokens.append(token)
        
        # Display token
        print(token, end='', flush=True)
    
    def on_llm_end(self, *args, **kwargs):
        """
        Called when LLM finishes generating.
        
        Display metrics if enabled.
        """
        end_time = time.time()
        
        if self.show_metrics:
            total_time = end_time - self.start_time
            first_token_latency = self.first_token_time - self.start_time if self.first_token_time else 0
            tokens_per_second = self.token_count / total_time if total_time > 0 else 0
            
            print(f"\n\n{self.prefix}[Streaming complete]")
            print(f"{self.prefix}Metrics:")
            print(f"{self.prefix}  - Tokens: {self.token_count}")
            print(f"{self.prefix}  - Total time: {total_time:.2f}s")
            print(f"{self.prefix}  - First token latency: {first_token_latency:.3f}s")
            print(f"{self.prefix}  - Tokens/second: {tokens_per_second:.1f}")
    
    def on_llm_error(self, error: Exception, **kwargs):
        """
        Called when LLM encounters an error.
        
        Args:
            error: Exception that occurred
        """
        print(f"\n\n{self.prefix}[Error during streaming: {str(error)}]", flush=True)
    
    def get_full_response(self) -> str:
        """
        Get the complete response text.
        
        Returns:
            Full response as string
        """
        return ''.join(self.tokens)

# Test custom callback
print("=== Custom Streaming Callback Demo ===")
print("Prompt: Explain how RAG systems work\n")

callback = CustomStreamingCallback(prefix="  ", show_metrics=True)

# Simulate streaming with callback
callback.on_llm_start()
for token in llm.stream("Explain RAG"):
    callback.on_llm_new_token(token)
callback.on_llm_end()

# Get full response
full_response = callback.get_full_response()
print(f"\nFull response length: {len(full_response)} characters")

## Implementation: Buffered Streaming

Buffer tokens for smoother display and better performance.

In [ ]:
# Buffered streaming handler
class BufferedStreamingHandler:
    """
    Streaming handler with buffering for smoother display.
    
    Buffers tokens and flushes when:
    - Buffer reaches size limit
    - Sentence boundary detected
    - Timeout reached
    """
    
    def __init__(
        self,
        buffer_size: int = 5,
        flush_on_sentence: bool = True,
        timeout_ms: int = 100
    ):
        """
        Initialize buffered handler.
        
        Args:
            buffer_size: Number of tokens to buffer before flushing
            flush_on_sentence: Flush at sentence boundaries
            timeout_ms: Maximum time to buffer (milliseconds)
        """
        self.buffer_size = buffer_size
        self.flush_on_sentence = flush_on_sentence
        self.timeout_ms = timeout_ms
        
        # Buffer state
        self.buffer = []
        self.last_flush_time = time.time()
    
    def add_token(self, token: str) -> Optional[str]:
        """
        Add token to buffer and flush if needed.
        
        Args:
            token: Token to add
        
        Returns:
            Flushed content if buffer was flushed, None otherwise
        """
        self.buffer.append(token)
        
        # Check flush conditions
        should_flush = False
        
        # Condition 1: Buffer size limit
        if len(self.buffer) >= self.buffer_size:
            should_flush = True
        
        # Condition 2: Sentence boundary
        if self.flush_on_sentence and token.strip() in ['.', '!', '?']:
            should_flush = True
        
        # Condition 3: Timeout
        time_since_flush = (time.time() - self.last_flush_time) * 1000
        if time_since_flush >= self.timeout_ms:
            should_flush = True
        
        # Flush if needed
        if should_flush:
            return self.flush()
        
        return None
    
    def flush(self) -> str:
        """
        Flush buffer and return content.
        
        Returns:
            Buffered content as string
        """
        if not self.buffer:
            return ""
        
        # Get buffered content
        content = ''.join(self.buffer)
        
        # Clear buffer
        self.buffer = []
        self.last_flush_time = time.time()
        
        return content

# Test buffered streaming
print("=== Buffered Streaming Demo ===")
print("\nUnbuffered (token-by-token):")
for token in llm.stream("Test"):
    print(f"[{token}]", end='', flush=True)

print("\n\nBuffered (5 tokens at a time):")
handler = BufferedStreamingHandler(buffer_size=5, flush_on_sentence=False)
for token in llm.stream("Test"):
    flushed = handler.add_token(token)
    if flushed:
        print(f"[{flushed}]", end='', flush=True)

# Flush remaining
remaining = handler.flush()
if remaining:
    print(f"[{remaining}]", end='', flush=True)

print("\n\nNotice: Buffered version displays in chunks, reducing UI updates")

## Exercise: Build a Streaming Chat Interface

**Objective**: Create a simple streaming chat interface with conversation history.

In [ ]:
# Exercise: Streaming chat interface
class StreamingChatInterface:
    """Simple streaming chat interface."""
    
    def __init__(self, llm):
        """
        Initialize chat interface.
        
        Args:
            llm: Streaming LLM to use
        """
        self.llm = llm
        self.conversation_history = []
    
    def chat(self, user_message: str) -> str:
        """
        Send message and stream response.
        
        Args:
            user_message: User's message
        
        Returns:
            Complete response
        """
        # Add user message to history
        self.conversation_history.append({"role": "user", "content": user_message})
        
        # Display user message
        print(f"\n👤 User: {user_message}")
        print("🤖 Assistant: ", end='', flush=True)
        
        # Stream response
        response_tokens = []
        for token in self.llm.stream(user_message):
            print(token, end='', flush=True)
            response_tokens.append(token)
        
        # Get complete response
        response = ''.join(response_tokens)
        
        # Add to history
        self.conversation_history.append({"role": "assistant", "content": response})
        
        print()  # New line after response
        return response
    
    def get_history(self) -> list:
        """Get conversation history."""
        return self.conversation_history.copy()

# Test chat interface
print("=== Streaming Chat Interface ===")
chat = StreamingChatInterface(llm)

# Simulate conversation
chat.chat("What is RAG?")
chat.chat("How does it work?")

print(f"\nConversation turns: {len(chat.get_history())}")

## Checkpoint: Self-Assessment

Before proceeding, ensure you can answer:

1. ✅ What is the main benefit of streaming responses?
   - Lower perceived latency, better user experience

2. ✅ What is time-to-first-token (TTFT)?
   - Time from request to first token received

3. ✅ Why use buffering in streaming?
   - Reduce UI updates, smoother display, better performance

4. ✅ When should you flush a buffer?
   - Buffer size limit, sentence boundary, timeout

5. ✅ How do callbacks enable streaming?
   - Called for each token, allowing incremental processing

## Troubleshooting: Common Streaming Issues

### Issue 1: Buffering Delays
**Problem**: Tokens don't appear immediately
**Solution**:
- Use `flush=True` in print statements
- Disable output buffering
- Check network buffering settings

### Issue 2: High Latency
**Problem**: Slow time-to-first-token
**Solution**:
- Optimize prompt length
- Use faster models for initial response
- Implement caching

### Issue 3: Connection Drops
**Problem**: Streaming interrupted mid-response
**Solution**:
- Implement reconnection logic
- Save partial responses
- Provide resume capability

## Next Steps

Continue to:
- **Module 3 Notebooks**: Evaluation and tuning
- **Module 4 Notebooks**: Knowledge integration and RAG

## References

- Course Notes: module-02-agent-development.md
- LangChain Streaming Documentation
- NVIDIA NIM Streaming Best Practices
- Server-Sent Events (SSE) Specification